# 모의고사 01. Titanic 생존자 예측 (이진분류)

> 실제 시험처럼 이론 설명 없이 문제만 순서대로 제시합니다. **90분 타이머를 맞추고** 풀어보세요. 오픈북 허용 문서(numpy/pandas/matplotlib/seaborn/scikit-learn/tensorflow/XGBoost 공식문서)만 참고할 수 있다고 가정합니다.

### 데이터 소개
`data/train.csv` - Titanic 승객 정보로 생존 여부(Survived, 0/1)를 예측합니다. 이번 회차는 `Name`에서 호칭(Title)을 뽑아내는 파생변수 문제가 포함됩니다.

> ⏱️ **[실전 타임어택 가이드]** 데이터 탐색 10분 ➔ 전처리 20분 ➔ 머신러닝 30분 ➔ 딥러닝 30분
> 막히는 부분은 주석으로 남기고 다음 문제로 넘어가는 것이 합격 전략입니다.


## 목차
1교시 데이터 로딩 & EDA · 2교시 데이터 시각화 · 3교시 데이터 전처리 · 4교시 머신러닝 모델링 · 5교시 딥러닝 모델링 · 채점 가이드

In [ ]:
import sys
sys.path.insert(0, '../00_시작하기')
import aice_grader as grader

# 실전처럼 시간 제한(90분)을 두고 풀어보세요. 아래 셀을 실행하면 타이머가 시작됩니다.
timer = grader.ExamTimer(minutes=90)
timer.start()

## 1교시: 데이터 로딩 & EDA

**문제 1.** `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`을 각각 `pd`, `np`, `plt`, `sns`로 불러오고, `../data/train.csv`을 `df`로 불러와 shape을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/train.csv')
print(df.shape)
```

</details>

**문제 2.** `df`의 shape과 컬럼별 결측치 개수를 출력하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(df.shape)  # (행 개수, 열 개수)로 전체 데이터 크기를 먼저 확인
print(df.isnull().sum())
```

</details>

**문제 3.** `Pclass`별 승객 수를 `value_counts()`로 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(df['Pclass'].value_counts())  # 값별 개수를 많은 순서로 정렬해 반환
```

</details>

**문제 4.** `Sex`와 `Survived`의 교차표를 `pd.crosstab`으로 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(pd.crosstab(df['Sex'], df['Survived']))  # 두 범주형 변수의 조합별 빈도를 표로 한 번에 확인
```

</details>

## 2교시: 데이터 시각화

**문제 5.** `Pclass`별 생존율을 bar 그래프로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df.groupby('Pclass')['Survived'].mean().plot(kind='bar')  # Survived가 0/1이라 평균을 내면 곧 생존율이 됨
plt.title('Survival rate by Pclass')
plt.show()
```

</details>

**문제 6.** `Age` 분포를 히스토그램(`kde=True`)으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.histplot(df['Age'].dropna(), kde=True)  # Age에 결측치가 있으면 히스토그램이 왜곡될 수 있어 dropna()로 미리 제외
plt.title('Age distribution')
plt.show()
```

</details>

**문제 7.** 수치형 변수 간 상관관계 heatmap을 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm')  # 문자열 컬럼은 상관계수를 구할 수 없어 수치형만 먼저 선택
plt.show()
```

</details>

## 3교시: 데이터 전처리

**문제 8.** `Name`에서 호칭(Title)을 추출해 `Title` 컬럼을 만드세요. (예: 'Braund, Mr. Owen Harris' -> 'Mr')

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df['Title'] = df['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())  # 'Braund, Mr. Owen' -> ','로 나눠 성 제외 -> '.'로 나눠 호칭만 추출 -> 공백 제거
print(df['Title'].value_counts())
```

</details>

**문제 9.** `Age`는 중앙값으로, `Embarked`는 최빈값으로 결측치를 채우세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df['Age'] = df['Age'].fillna(df['Age'].median())  # 수치형은 이상치 영향이 적은 중앙값으로 대체
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])  # 범주형은 최빈값(mode)으로 대체, mode()는 Series라서 [0]으로 첫 값을 꺼냄
```

</details>

**문제 10.** `Sex`, `Embarked`를 `get_dummies(drop_first=True)`로 인코딩하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)  # drop_first=True로 다중공선성을 피하면서 원-핫 인코딩
df.head()
```

</details>

**문제 11.** `Pclass`, `Age`, `SibSp`, `Parch`, `Fare`, `Sex_male`, 인코딩된 `Embarked_*` 컬럼으로 X, `Survived`로 y를 만들고 `train_test_split(test_size=0.2, stratify=y, random_state=42)`로 나누세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import train_test_split
feature_cols = ['Pclass','Age','SibSp','Parch','Fare','Sex_male'] + [c for c in df.columns if c.startswith('Embarked_')]  # get_dummies가 만든 'Embarked_Q' 같은 컬럼명은 startswith로 찾아 합침
X = df[feature_cols]
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # stratify=y로 생존 비율을 train/test에 동일하게 유지
print(X_train.shape)
```

</details>

## 4교시: 머신러닝 모델링

**문제 12.** `RandomForestClassifier(n_estimators=100, random_state=42)`를 학습시키고 accuracy, f1-score를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
model = RandomForestClassifier(n_estimators=100, random_state=42)  # 여러 트리를 배깅 방식으로 묶어 단일 트리보다 안정적인 성능을 냄
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(accuracy_score(y_test, pred))
print(f1_score(y_test, pred))  # 이진분류라 average 옵션 없이 바로 계산됨
```

</details>

**문제 13.** confusion matrix를 heatmap으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')  # fmt='d'로 칸 안의 숫자를 정수로 표시
plt.show()
```

</details>

**문제 14.** `model`의 feature_importances_ 중 가장 중요한 변수를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
imp = pd.Series(model.feature_importances_, index=X.columns)  # feature_importances_ 배열에 컬럼명을 붙여 어떤 값이 어떤 변수인지 알 수 있게 함
print(imp.idxmax())  # 값이 아니라 값이 가장 큰 위치(컬럼명)를 반환
```

</details>

## 5교시: 딥러닝 모델링

**문제 15.** StandardScaler로 X_train/X_test를 스케일링한 뒤, 출력층 1노드(sigmoid)의 Sequential 모델을 만들어 `binary_crossentropy`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # train에만 fit
X_test_s = scaler.transform(X_test)  # test는 transform만(학습 시 보지 못한 정보가 새어 들어가는 데이터 누수 방지)
dl_model = Sequential()
dl_model.add(Dense(16, activation='relu', input_shape=(X_train_s.shape[1],)))
dl_model.add(Dense(8, activation='relu'))
dl_model.add(Dense(1, activation='sigmoid'))  # 이진분류 출력층: 노드 1개 + sigmoid
dl_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
```

</details>

**문제 16.** `EarlyStopping(patience=10)`을 사용해 `epochs=100`으로 학습시키세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)  # val_loss가 10 epoch 동안 개선 없으면 학습을 멈추고 가장 좋았던 가중치로 복원
history = dl_model.fit(X_train_s, y_train, epochs=100, batch_size=32, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
```

</details>

**문제 17.** 테스트 데이터의 accuracy를 `evaluate()`로 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
loss, acc = dl_model.evaluate(X_test_s, y_test, verbose=0)  # evaluate()는 compile 시 지정한 순서대로 (loss, metrics...)를 반환
print(acc)
```

</details>

In [ ]:
# 문제를 다 풀었다면 아래 셀을 실행해 소요 시간을 확인하세요.
timer.finish()

## 채점 가이드 (100점 만점 배점표)

이 모의고사는 총 17문제, 100점 만점입니다. 각 문제를 정답과 비교해 맞았으면 배점만큼, 틀렸으면 0점으로 스스로 채점해보세요.

| 문항 | 배점 |
|---|---|
| 문제 1 | 5점 |
| 문제 2 | 5점 |
| 문제 3 | 6점 |
| 문제 4 | 6점 |
| 문제 5 | 6점 |
| 문제 6 | 6점 |
| 문제 7 | 6점 |
| 문제 8 | 6점 |
| 문제 9 | 6점 |
| 문제 10 | 6점 |
| 문제 11 | 6점 |
| 문제 12 | 6점 |
| 문제 13 | 6점 |
| 문제 14 | 6점 |
| 문제 15 | 6점 |
| 문제 16 | 6점 |
| 문제 17 | 6점 |
| **합계** | **100점** |

> 💡 **합격 기준: 80점 이상** (실제 AICE Associate와 동일한 기준입니다)

### 자동으로 기록하며 채점하고 싶다면

`00_시작하기/aice_grader.py`의 `MockExamGrader`를 사용하면 점수를 자동으로 합산해줍니다.

```python
import aice_grader as grader

g = grader.MockExamGrader("모의고사01_Titanic_생존자예측")
g.check(1, points=5, is_correct=True)   # 문제 1을 맞았으면 True, 틀렸으면 False
g.check(2, points=5, is_correct=True)
# ... 문제 수만큼 반복 ...
g.report()   # 최종 점수와 합격 여부를 출력
```

### 개념 체크리스트

다 풀었다면 아래 체크리스트로 한 번 더 점검해보세요.

- [ ] 라이브러리를 정확한 alias로 불러왔는가
- [ ] 결측치를 모두 처리했는가
- [ ] 인코딩/스케일링을 빠뜨리지 않았는가
- [ ] 이진분류 평가지표(accuracy, f1)를 모두 계산했는가
- [ ] 출력층 activation을 'sigmoid'로 지정했는가
